In [21]:
### Chunk 1: Imports & Data Loading / Scaling

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold

# --- Load & scale training data ---
train_df = pd.read_csv(
    'C:/Users/dash0006/OneDrive - University of Oklahoma/'
    'MVGPR_SS_Nano_p2/Python_code/Data/dat_bin_odd_c.csv'
)
X_train = train_df[['Mo', 'Nb', 'Ta', 'V', 'W']].values.astype(np.float32)
y_train = train_df[
    ['m_log_max_strain', 'm_bc_1', 'm_bc_2', 'm_bc_3', 'm_bc_4']
].values.astype(np.float32)

input_scaler = MinMaxScaler(feature_range=(0, 1))
output_scaler = MinMaxScaler(feature_range=(0, 1))
X_train_scaled = input_scaler.fit_transform(X_train)
y_train_scaled = output_scaler.fit_transform(y_train)

In [22]:
### Chunk 2: Training Helper & FlexibleNet Definition

def train_model(model, optimizer, criterion, loader, num_epochs):
    model.train()
    for epoch in range(num_epochs):
        losses = []
        for xb, yb in loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {np.mean(losses):.6f}")
    return np.mean(losses)

class FlexibleNet(nn.Module):
    def __init__(
        self,
        input_size=5,
        hidden_size=32,
        num_layers=4,
        dropout_rate=0.2,
        output_size=5
    ):
        super().__init__()
        layers = []
        for i in range(num_layers):
            in_feats = input_size if i == 0 else hidden_size
            layers += [
                nn.Linear(in_feats, hidden_size),
                nn.ReLU(),
                nn.Dropout(dropout_rate)
            ]
        layers.append(nn.Linear(hidden_size, output_size))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

In [23]:
### Chunk 3 (updated): Full 4‐D Grid Search

def grid_search_full(
    X, y,
    layer_list, neuron_list,
    lr_list, batch_size_list,
    num_epochs=20, n_splits=5
):
    # stratify on primary target
    bins = pd.qcut(y[:,0], q=n_splits, labels=False, duplicates='drop')
    skf  = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    best = {'val_loss': float('inf')}
    results = []

    for nl in layer_list:
        for nh in neuron_list:
            for lr in lr_list:
                for bs in batch_size_list:
                    fold_losses = []
                    print(f"\nTesting → layers={nl}, neurons={nh}, lr={lr}, batch_size={bs}")
                    for tr_idx, val_idx in skf.split(X, bins):
                        X_tr, y_tr = X[tr_idx], y[tr_idx]
                        X_val, y_val = X[val_idx], y[val_idx]

                        tr_ds = TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr))
                        tr_loader = DataLoader(tr_ds, batch_size=bs, shuffle=True)

                        model = FlexibleNet(
                            input_size=X.shape[1],
                            hidden_size=nh,
                            num_layers=nl,
                            dropout_rate=0.2,
                            output_size=y.shape[1]
                        )
                        criterion = nn.MSELoss()
                        optimizer = optim.Adam(model.parameters(), lr=lr)

                        # train one fold
                        train_model(model, optimizer, criterion, tr_loader, num_epochs)

                        # validate
                        model.eval()
                        with torch.no_grad():
                            val_preds = model(torch.tensor(X_val))
                            loss_val  = criterion(val_preds, torch.tensor(y_val)).item()
                        fold_losses.append(loss_val)

                    avg_loss = np.mean(fold_losses)
                    print(f"→ avg val loss: {avg_loss:.6f}")
                    results.append({
                        'layers': nl,
                        'neurons': nh,
                        'lr': lr,
                        'batch_size': bs,
                        'val_loss': avg_loss
                    })
                    if avg_loss < best['val_loss']:
                        best = {
                            'layers': nl,
                            'neurons': nh,
                            'lr': lr,
                            'batch_size': bs,
                            'val_loss': avg_loss
                        }

    print(
        f"\nBest config → layers={best['layers']}, neurons={best['neurons']}, "
        f"lr={best['lr']}, batch_size={best['batch_size']} "
        f"(val_loss={best['val_loss']:.6f})"
    )
    return best, results


In [24]:
### Chunk 4 (updated): Run Full Grid Search & Train Final Model

# define your grids
layer_list      = list(range(3, 7))
neuron_list     = [4,8,16,32,64]
lr_list         = [0.001, 0.005, 0.01]
batch_size_list = [2, 4, 8, 16]

# run 4-D grid search
best_cfg, all_results = grid_search_full(
    X_train_scaled, y_train_scaled,
    layer_list, neuron_list,
    lr_list, batch_size_list,
    num_epochs=20,
    n_splits=5
)

# unpack best
nl = best_cfg['layers']
nh = best_cfg['neurons']
lr = best_cfg['lr']
bs = best_cfg['batch_size']

# final training on full data
final_model = FlexibleNet(
    input_size=5,
    hidden_size=nh,
    num_layers=nl,
    dropout_rate=0.2,
    output_size=5
)
optimizer = optim.Adam(final_model.parameters(), lr=lr)
criterion = nn.MSELoss()

full_ds     = TensorDataset(
    torch.tensor(X_train_scaled),
    torch.tensor(y_train_scaled)
)
full_loader = DataLoader(full_ds, batch_size=bs, shuffle=True)

print("\nTraining final model on full training set...")
final_loss = train_model(final_model, optimizer, criterion, full_loader, num_epochs=20)
print(f"Final training loss: {final_loss:.6f}")



Testing → layers=3, neurons=4, lr=0.001, batch_size=2
Epoch 1/20, Loss: 0.187169
Epoch 2/20, Loss: 0.165095
Epoch 3/20, Loss: 0.150274
Epoch 4/20, Loss: 0.152495
Epoch 5/20, Loss: 0.115597
Epoch 6/20, Loss: 0.129293
Epoch 7/20, Loss: 0.112532
Epoch 8/20, Loss: 0.099801
Epoch 9/20, Loss: 0.109628
Epoch 10/20, Loss: 0.087236
Epoch 11/20, Loss: 0.096177
Epoch 12/20, Loss: 0.079314
Epoch 13/20, Loss: 0.074559
Epoch 14/20, Loss: 0.066844
Epoch 15/20, Loss: 0.074507
Epoch 16/20, Loss: 0.072145
Epoch 17/20, Loss: 0.083664
Epoch 18/20, Loss: 0.069060
Epoch 19/20, Loss: 0.080860
Epoch 20/20, Loss: 0.089756
Epoch 1/20, Loss: 0.256836
Epoch 2/20, Loss: 0.240645
Epoch 3/20, Loss: 0.219639
Epoch 4/20, Loss: 0.207492
Epoch 5/20, Loss: 0.196505
Epoch 6/20, Loss: 0.180656
Epoch 7/20, Loss: 0.170962
Epoch 8/20, Loss: 0.161977
Epoch 9/20, Loss: 0.163981
Epoch 10/20, Loss: 0.150293
Epoch 11/20, Loss: 0.143867
Epoch 12/20, Loss: 0.141325
Epoch 13/20, Loss: 0.135577
Epoch 14/20, Loss: 0.125341
Epoch 15/20

In [25]:
import os
from sklearn.metrics import r2_score, mean_squared_error

# Directory and test files
data_dir = 'C:/Users/dash0006/OneDrive - University of Oklahoma/MVGPR_SS_Nano_p2/Python_code/Data/May-Results'
test_files = [
    'dat_test_bin_even_c.csv',
    'dat_test_qq_c.csv'
]

input_cols = ['Mo', 'Nb', 'Ta', 'V', 'W']
output_names = ['m_log_max_strain', 'm_bc_1', 'm_bc_2', 'm_bc_3', 'm_bc_4']

for fname in test_files:
    path = os.path.join(data_dir, fname)
    df = pd.read_csv(path)
    
    # Extract and scale inputs
    X_test = df[input_cols].values.astype(np.float32)
    X_test_scaled = input_scaler.transform(X_test)
    X_test_tensor = torch.tensor(X_test_scaled)
    
    # Predict (scaled) and invert transform
    final_model.eval()
    with torch.no_grad():
        preds_scaled = final_model(X_test_tensor).numpy()
    preds = output_scaler.inverse_transform(preds_scaled)
    
    # Save inputs + predictions
    pred_df = df[input_cols].copy()
    for i, name in enumerate(output_names):
        pred_df[name] = preds[:, i]
    out_fname = f'NEWpredictions_{os.path.splitext(fname)[0]}.csv'
    out_path = os.path.join(data_dir, out_fname)
    pred_df.to_csv(out_path, index=False)
    print(f"Saved predictions to: {out_path}")
    
    # Compute metrics against the true outputs in df
    actual = df[output_names].values.astype(np.float32)
    mask = ~np.isnan(actual).any(axis=1)
    actual_clean, preds_clean = actual[mask], preds[mask]
    
    print(f"\nMetrics for {fname}:")
    for i, name in enumerate(output_names):
        r2 = r2_score(actual_clean[:, i], preds_clean[:, i])
        rmse = mean_squared_error(actual_clean[:, i], preds_clean[:, i], squared=False)
        print(f"  {name}: R² = {r2:.4f}, RMSE = {rmse:.4f}")
    print("-" * 40)

Saved predictions to: C:/Users/dash0006/OneDrive - University of Oklahoma/MVGPR_SS_Nano_p2/Python_code/Data/May-Results\NEWpredictions_dat_test_bin_even_c.csv

Metrics for dat_test_bin_even_c.csv:
  m_log_max_strain: R² = 0.7349, RMSE = 0.5039
  m_bc_1: R² = 0.9076, RMSE = 0.2851
  m_bc_2: R² = 0.9103, RMSE = 0.2765
  m_bc_3: R² = 0.7196, RMSE = 0.4356
  m_bc_4: R² = 0.7880, RMSE = 0.4089
----------------------------------------
Saved predictions to: C:/Users/dash0006/OneDrive - University of Oklahoma/MVGPR_SS_Nano_p2/Python_code/Data/May-Results\NEWpredictions_dat_test_qq_c.csv

Metrics for dat_test_qq_c.csv:
  m_log_max_strain: R² = 0.3795, RMSE = 0.2010
  m_bc_1: R² = 0.0407, RMSE = 0.4371
  m_bc_2: R² = 0.0776, RMSE = 0.4082
  m_bc_3: R² = -0.2225, RMSE = 0.2623
  m_bc_4: R² = 0.0293, RMSE = 0.3440
----------------------------------------


C:\Users\dash0006\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\dash0006\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\dash0006\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\dash0006\AppData\Local\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will b